<h1><center>Laboratorio 5: La desperación de Mr. Lepin 🐼</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes


### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Javier Cruz Araneda
- Nombre de alumno 2: Enzo Toledo Venegas


---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Antigravity, Cursor, etc.) restringido a consultas, documentación y corrección de errores. 
- **Importante**: **¡Recuerden fijar semillas!** Así podemos reproducir sus resultados.

## Descripción del laboratorio.

### Importamos librerias utiles 😸

In [1]:
!uv add numpy pandas scikit-learn umap-learn plotly

Resolved 124 packages in 0.88ms
Checked 120 packages in 170ms


In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.base import BaseEstimator, TransformerMixin


def plot_dim_reductions(
    pca_proj: np.ndarray,
    tsne_proj: np.ndarray,
    umap_proj: np.ndarray,
    name: None | str = None,
    colors: None | np.ndarray = None,
) -> go.Figure:
    fig = make_subplots(rows=1, cols=3, subplot_titles=("PCA", "t-SNE", "UMAP"))

    for i, (proj, title) in enumerate(zip([pca_proj, tsne_proj, umap_proj], ["PCA", "t-SNE", "UMAP"], strict=True)):
        temp_fig = px.scatter(
            x=proj[:, 0],
            y=proj[:, 1],
            color=colors.astype(str) if colors is not None else None,
            title=title,
            # showlegend=(i == 0),
        )

        for trace in temp_fig.data:
            trace.showlegend = i == 0
            fig.add_trace(trace, row=1, col=i + 1)

    fig.update_layout(height=400, width=1200, title_text=name)
    return fig

# Segmentación de Clientes en Tienda de Retail 🛍️

<p align="center">
  <img width=300 src="https://s1.eestatic.com/2018/04/14/social/la_jungla_-_social_299733421_73842361_854x640.jpg">
</p>

## 1.1 Cargar Dataset

Mr. Lepin, en una nueva reunión, le cuenta a ud y su equipo que los resultados derivados del análisis exploratorio de datos presentaron una gran utilidad para la empresa y que tiene un gran entusiasmo por continuar trabajando con ustedes.
Es por esto, que Mr. Lepin les pide que cargue y visualicen algunas de las filas que componen el Dataset.
A continuación un extracto de lo parlamentado en la reunión:

    - Usted: Es un gran logro para nuestro equipo que usted haya encontrado excelente el EDA. ¿Qué tiene en mente ahora?
    - Mr. Lepin: Resulta que hace algún tiempo, mientras tomaba un mojito en una reunión de gerentes en Panamá, oí a un *chato* acerca de **LRMFP**, que es un modelo que permite personificar a los clientes a través de la fabricación de distintos atributos que describen a los clientes. Lo encontré es-tu-pendo ñatito. 
    - Usted: Ehh bueno. Investigaremos acerca de este modelo y veremos lo que podemos hacer.

Por ende, su siguiente tarea es calcular **LRMFP** sobre cada cliente y luego hacer un análisis de las características generadas. Para esto, el área de ventas les entrega un nuevo archivo llamado `retail_dataset.pickle`, quien posee los datos del DataFrame original limpios y listos para obtener las características solicitadas por Mr. Lepin.

In [3]:
df_retail = pd.read_pickle(
    "https://github.com/MDS7202/MDS7202/raw/refs/heads/main/recursos/2026-01/labs/lab6/retail_dataset.pickle"
)
df_retail = df_retail.astype(
    {
        "Invoice": str,
        "StockCode": str,
        "Description": str,
        "Customer ID": str,
        "Country": str,
    }
)
df_retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
# Trabajaremos con una copia del dataframe original para evitar modificarlo directamente:
df_copy = df_retail.copy()
df_copy.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 1.2 Creación de nuevas Caracteristicas [2 Puntos] 

Como ya se les comentó, Mr. Lepin está interesado en obtener las características **LRMFP**, para esto les señala que estas características se construyen en base a las siguientes definiciones:

- **Length (L)**: Intervalo de tiempo, en días, entre la primera y la última visita del cliente. Mientras más grande sea el valor, más fiel es el cliente.

- **Recency (R)**: Indica hace cuánto tiempo el cliente realizó su última compra. Notar que para este caso, mientras más grande es el valor, menos interés posee el usuario para repetir una compra en uno de los locales.

- **Monetary (M)**: El término “monetario” se refiere a la cantidad media de dinero gastada por cada visita del cliente durante el período de observación y refleja la contribución del cliente a los ingresos de la empresa.

- **Frequency (F)**: Se refiere al número total de visitas del cliente durante el periodo de observación. Cuanto mayor sea la frecuencia, mayor será la fidelidad del cliente. 

- **Periodicity (P)**: Representa si los clientes visitan las tiendas con regularidad.

$$Periodicity(n)=std(IVT_1, ..., IVT_n)$$

&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Donde $IVT$ denota el tiempo entre visitas y n representa el número de valores de tiempo entre visitas de un cliente.
 

$$IVT_i=date\_diff(t_{i+1},t_i)$$

En base a las definiciones señaladas, diseñe una función que permita obtener las características **LRMFP** recibiendo un DataFrame como entrada. Para esto, no estará permitido el uso de iteradores; utilicen todas las herramientas que les ofrece `pandas` para realizar esto.

Una referencia que les puede ser útil es el [documento original](https://www.researchgate.net/publication/315979555_LRFMP_model_for_customer_segmentation_in_the_grocery_retail_industry_a_case_study) en donde se propone este método.

**<u>Formato</u> del Resultado Esperado:**

| Customer ID | Length | Recency | Frequency | Monetary | Periodicity |
|------------:|-------:|--------:|----------:|---------:|------------:|
|   12346.0   |    294 |      67 |        46 |   -64.68 |        37.0 |
|   12347.0   |     37 |       3 |        71 |  1323.32 |         0.0 |
|   12349.0   |    327 |      43 |       107 |  2646.99 |        78.0 |
|   12352.0   |     16 |      11 |        18 |   343.80 |         0.0 |
|   12356.0   |     44 |      16 |        84 |  3562.25 |        12.0 |

**Respuesta:**

In [5]:
def custom_features(dataframe_in: pd.DataFrame) -> pd.DataFrame:
    """
    Recibe un dataframe y devuelve otro nuevo con las características LRMFP
    """
    # Crear dataframe de salida y copia de input para trabajar:
    df = pd.DataFrame()
    df_in = dataframe_in.copy()

    # Length (L):
    customer_final = df_in.groupby("Customer ID")["InvoiceDate"].max()
    customer_initial = df_in.groupby("Customer ID")["InvoiceDate"].min()
    df["Length"] = (customer_final - customer_initial).dt.days

    # Recency (R):
    final_date = df_in["InvoiceDate"].max()
    df["Recency"] = (final_date - customer_final).dt.days + 1  # debemos sumar 1 pues en la resta se pierde un día.

    # Monetary (M)
    df_in["TotalSpent"] = df_in["Price"] * df_in["Quantity"]
    # OBSERVACION: Tengo dudas con esto... Así como está hecho ahora da igual que la tabla que pusieron arriba,
    # Efectivamente, tras confirmar por correo, el ultimo .sum() debe ser .mean():
    df["Monetary"] = (
        df_in.groupby(["Customer ID", "Invoice"])["TotalSpent"].sum().groupby("Customer ID").mean().round(2)
    )

    # Frequency (F):
    # df["Frequency"] = df_in.groupby("Customer ID")["Invoice"].nunique()
    # Tras verificar por correo, esta es la forma de calcular deseada:
    df["Frequency"] = df_in.groupby("Customer ID")["Invoice"].count()

    # Periodicty (P):
    # Eliminamos duplicados para quedarnos con las fechas únicas por visita y cliente
    visitas = df_in[["Customer ID", "Invoice", "InvoiceDate"]].drop_duplicates()

    # Ordenar para: (t_i+1 - t_i)
    visitas = visitas.sort_values(by=["Customer ID", "InvoiceDate"])

    # Calcular la diferencia en días con la visita anterior
    visitas["IVT"] = visitas.groupby("Customer ID")["InvoiceDate"].diff().dt.days

    # Obtener Periodicity:
    df["Periodicity"] = visitas.groupby("Customer ID")["IVT"].std().fillna(0).round(1)

    return df

In [6]:
df = custom_features(df_copy)
customer_tests = ["12346.0", "12347.0", "12349.0", "12352.0", "12356.0"]
df.loc[customer_tests]

,Length,Recency,Monetary,Frequency,Periodicity
Customer ID,,,,,
12346.0,196,165,33.90,33,36.7
12347.0,37,3,661.66,71,0.0
12349.0,181,43,890.38,102,101.8
12352.0,16,11,171.90,18,0.0
12356.0,44,16,1186.77,83,13.4


**<u>Formato</u> del Resultado Esperado:** (valores mal)

| Customer ID | Length | Recency | Frequency | Monetary | Periodicity |
|------------:|-------:|--------:|----------:|---------:|------------:|
|   12346.0   |    294 |      67 |        46 |   -64.68 |        37.0 |
|   12347.0   |     37 |       3 |        71 |  1323.32 |         0.0 |
|   12349.0   |    327 |      43 |       107 |  2646.99 |        78.0 |
|   12352.0   |     16 |      11 |        18 |   343.80 |         0.0 |
|   12356.0   |     44 |      16 |        84 |  3562.25 |        12.0 |

> **Observación Final:** 

> Tras consultar por correo, se confirmó que la forma adecuada de calcular Monetary es usando el .mean() y no el .sum() final, con lo que si bien no da igual que esta tabla anterior de referencia, está bien. Bajo esa misma lógica, algunos valores de esta tabla de referencia no coinciden, pero ahora todo debería estar bien tras las correciones y el correo enviado. Además, se optó por dejar las columnas ordenadas según las siglas del método, es decir, LRMFP.

## 1.3 Pipelines 👷

Finalmente *Mr. Lepin* le pregunta si sería posible realizar un pipeline para realizar una segmentación de los clientes con los nuevos datos generados, a lo que usted responde que **sí** y propone la utilización de k-means para la segmentación.

A continuación siga los pasos requeridos para obtener la segmentación de clientes.

### 1.3.1 Estandarizar Caracteristicas [0.5 puntos]

Construya una clase llamada ``MinMax()`` utilizando ``BaseEstimator`` y ``TransformerMixin`` para realizar una transformación de cada una de las columnas de un DataFrame utilizando ``ColumnTransformer()`` más tarde (tome como referencia el siguiente [enlace](https://sklearn-template.readthedocs.io/en/latest/user_guide.html#transformer)).


 Para esto considere que Min-Max escaler queda dada por la ecuación:

$$MinMax = \dfrac{x-min(x)}{max(x) - min(x)}$$

Con esto buscamos que los valores que componen a las columnas se muevan en el rango de valores $[0, 1]$.

**Respuesta:**

In [7]:
class MinMax(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Aprendemos el mínimo  máximo pues los necesitamos para luego transformar las cosas:
        self.min = X.min()
        self.max = X.max()

        # Por documentación fit retorna self (para dps encadenar con transform):
        return self

    def transform(self, X):
        # Evitemos divisiones por 0, recordando que la convención en ese caso es que el valor sea 1:
        range = self.max - self.min
        range = np.where(range == 0, 1, range)  # No sirve un if y reemplazar 1 pues tenemos series de pandas.
        # Aquí aplicamos la transformación usando los parámetros aprendidos en fit:
        X_scaled = (X - self.min) / range
        return X_scaled

### 1.3.2 Pipelines de Proyecciones [0.5 puntos]

Para comparar técnicas de reducción de dimensionalidad, realice **tres pipelines** distintos sobre los datos **LRMFP** usando los siguientes métodos:
- **PCA**
- **t-SNE**
- **UMAP**

Para cada pipeline, siga estos pasos:
1. Obtenga las características **LRMFP** desde el DataFrame `retail_dataset.pickle` utilizando la función ``custom_features`` creada anteriormente, junto a ``FunctionTransformer()``. Considere esto como el primer paso de su pipeline.
2. En segundo lugar, usando ``ColumnTransformer()``, aplique el MinMax scaler creado por usted sobre todas las columnas generadas en el paso anterior.
3. Finalmente, aplique el método de reducción de dimensionalidad correspondiente (PCA, t-SNE o UMAP) para obtener las 2 componentes más relevantes.

A continuación, grafique las proyecciones obtenidas de las tres técnicas en una sola figura comparativa.

**Respuesta:**

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from umap import UMAP

c:\Users\Javier\Desktop\MDS7202\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
features = ["Length", "Recency", "Monetary", "Frequency", "Periodicity"]
scaler_transformer = ColumnTransformer(transformers=[("minmax", MinMax(), features)])

# Fijaremos seed en 42 para reproducibilidad, aunque no es necesario para PCA, sí lo es para t-SNE y UMAP:
pipeline_pca = Pipeline(
    [
        ("features", FunctionTransformer(custom_features)),
        ("scaler", scaler_transformer),
        ("pca", PCA(n_components=2, random_state=42)),
    ]
)

pipeline_tsne = Pipeline(
    [
        ("features", FunctionTransformer(custom_features)),
        ("scaler", scaler_transformer),
        ("tsne", TSNE(n_components=2, random_state=42)),
    ]
)

pipeline_umap = Pipeline(
    [
        ("features", FunctionTransformer(custom_features)),
        ("scaler", scaler_transformer),
        ("umap", UMAP(n_components=2, random_state=42)),
    ]
)

In [10]:
# Utilice este código para ejecutar las pipelines y graficar.

pca_proj = pipeline_pca.fit_transform(df_retail)
tsne_proj = pipeline_tsne.fit_transform(df_retail)
umap_proj = pipeline_umap.fit_transform(df_retail)

fig = plot_dim_reductions(pca_proj, tsne_proj, umap_proj, name="Reducción de Dimensionalidad", colors=None)
fig.show()

c:\Users\Javier\Desktop\MDS7202\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


### 1.3.3 Análisis de los Loadings de PCA [0.5 puntos]
Antes de continuar con la etapa de clustering, analice los *loadings* (pesos o coeficientes) de las componentes principales obtenidas con PCA. 

Utilice el siguiente tutorial para visualizarlos: https://plotly.com/python/pca-visualization/

- Calcule y reporte los *loadings* de las dos primeras componentes principales.
- Interprete qué características (**LRMFP**) son más relevantes en cada componente.
- Visualice los *loadings* usando un gráfico de barras para cada componente.



In [11]:
# Código para calcular loadings.
pca_model = pipeline_pca.named_steps["pca"]
loadings = pca_model.components_
df_loadings = pd.DataFrame(
    loadings.T, columns=["PC1", "PC2"], index=["Length", "Recency", "Monetary", "Frequency", "Periodicity"]
)
df_loadings.head()

,PC1,PC2
Length,0.862777,0.447141
Recency,-0.464791,0.885060
Monetary,0.004563,-0.002213
Frequency,0.033931,0.009078
Periodicity,0.195995,0.129020


> **Observación:** Vemos que en ambas componentes principales PC1 y PC2 `Length` y `Recency` son las características más relevantes, siendo en PC1 `Length` la de mayor contribución y en PC2 `Recency`. 

In [12]:
fig = make_subplots(
    rows=1, cols=2, subplot_titles=("Loadings de Componente Principal 1", "Loadings de Componente Principal 2")
)

# Agregar Barras para PC1
fig.add_trace(
    go.Bar(x=df_loadings.index, y=df_loadings["PC1"], name="PC1", marker_color="rgb(55, 83, 109)"), row=1, col=1
)

# Agregar Barras para PC2
fig.add_trace(
    go.Bar(x=df_loadings.index, y=df_loadings["PC2"], name="PC2", marker_color="rgb(26, 118, 255)"), row=1, col=2
)

fig.update_layout(title="Análisis de Loadings PCA", showlegend=False, height=500)
fig.show()

In [13]:
# Visualización de Loadings como Vectores en el Espacio PCA:
fig = go.Figure()
for feature in df_loadings.index:
    fig.add_trace(
        go.Scatter(
            x=[0, df_loadings.loc[feature, "PC1"]],
            y=[0, df_loadings.loc[feature, "PC2"]],
            mode="lines+markers+text",
            text=[None, feature],
            textposition="top center",
            name=feature,
        )
    )
fig.update_layout(title="Loadings PCA como Vectores", xaxis_title="PC1", yaxis_title="PC2", showlegend=False)
fig.show()

In [14]:
import pandas as pd

# Creamos la figura dispersando las dos componentes (x es la columna 0, y la 1)
# Pintaremos según la columna 'Monetary' usando una escala de colores secuencial
fig = px.scatter(
    x=pca_proj[:, 0],
    y=pca_proj[:, 1],
    color=np.log1p(df["Monetary"]),  # usamos log para que se vea mejor pese a outliers o valores muy grandes
    color_continuous_scale="Viridis",
    labels={"x": "Componente 1 (PC1)", "y": "Componente 2 (PC2)", "color": "Gasto Total"},
    title="Clientes (PCA) coloreados por Valor Monetario (M)",
    opacity=0.8,
)

fig.show()

### Preguntas sobre loadings:

- ¿Qué son los loadings de PCA?

> Respuesta: Los loadings son los pesos de la transformación hecha para obtener las componentes principales con respecto a las características originales. Estos indican que tanto contribuye cada variable original en la creación de cada componente, variando entre -1 y 1 y donde su importancia viene dada por el valor absoluto de su valor. Una valor absoluto grande indica alta influencia en esa componente y el signo entrega información de la correlación de la variable con respecto al eje principal donde está proyectada.

- ¿Qué información relevante obtiene sobre la estructura de los datos a partir de los *loadings* de PCA?

> Respuesta: En ambas componentes (PC1 y PC2) observamos que las variables más determinantes en el comportamiento son `Length` y `Recency`, siendo en PC1 `Length` la de mayor contribución y en PC2 `Recency`. Algo interesante es que variables que podrían ser intuitivamente importantes no tienen tanto peso en ambas componentes como `Monetary` relacionado al gasto de dinero y `Frequency` junto a `Periodicty` relacionadas a la regularidad y consistencia de visitas de clientes.

- ¿Existe alguna relación interesante entre las direcciones de las variables?

> Respuesta: Sí, por ejemplo, en PC1 `Length` y `Recency` presentan una correlación inversa, esto tiene sentido si pensamos que un `Length` alto indica un cliente fiel, por tanto, es más probable que vuelva regularmente a comprar por lo que su tiempo entre compras tiende a ser menor.  

## 1.4 Clustering

### 1.4.1 Método del Codo [0.5 puntos]

Utilizando la clase creada para escalamiento, aplique el método del codo para visualizar cuál es el número de clusters que mejor se ajustan a los datos. Realice esto utilizando el algoritmo K-means dentro de un pipeline para un $k \in [1,20]$, donde k representa el número de clusters del k-means. Para la realización de esta sección y la próxima (1.4.2), considere los mismos pasos utilizados para el t-SNE, pero **permutando el algoritmo de reducción de dimensionalidad por k-means.**

**Respuesta:**

In [15]:
from sklearn.cluster import KMeans  # noqa: F401

# Calculamos la inercia para diferentes valores de k
inertias = []
k_range = range(1, 21)

for k in k_range:
    pipeline_kmeans = Pipeline(
        [
            ("features", FunctionTransformer(custom_features)),
            ("scaler", scaler_transformer),
            ("kmeans", KMeans(n_clusters=k, random_state=42, n_init=10)),
        ]
    )

    pipeline_kmeans.fit(df_retail)
    kmeans_model = pipeline_kmeans.named_steps["kmeans"]
    inertias.append(kmeans_model.inertia_)

In [16]:
# Graficamos el método del codo
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=list(k_range),
        y=inertias,
        mode="lines+markers",
        name="Inercia",
        marker=dict(size=8, color="rgb(55, 83, 109)"),
        line=dict(color="rgb(55, 83, 109)", width=2),
    )
)

fig.update_layout(
    title="Método del Codo para K-Means",
    xaxis_title="Número de Clusters (k)",
    yaxis_title="Inercia",
    height=500,
    width=800,
    hovermode="x unified",
)
fig.show()

### Preguntas Método del Codo

- A través del gráfico obtenido, comente y justifique qué valor de k escogería para realizar el k-means.

> Respuesta: 

Según el gráfico obtenido, creemos que el valor óptimo es $k=3$, ya que observamos que la inercia a partir de este punto decrece de forma más suave. Es decir, cada vez que aumentemos la complejidad del modelo añadiendole más cantidad de clusters, el error que logramos corregir es despreciable. 

- Le fue útil el método del codo para encontrar el número de clusters?

> Respuesta:

Sí, en este caso el método fue efectivo debido a la presencia de un punto de inflexión claro en la curva de inercia, lo que permitió identificar el número óptimo de clusters de manera objetiva. Sin embargo, cabe destacar que en escenarios más complejos, por ejemplo, al manejar datos de alta dimensionalidad, este método puede volverse ambiguo al no presentar un "codo" definido.

- Si no fue así, ¿qué otros métodos podría haber usado para encontrar un número óptimo de clusters?
 
> Respuesta:

En caso de que el codo no fuera claro, se puede recurrir al Análisis de Silueta (Silhouette Score), que mide qué tan bien agrupado está cada dato comparando su cercanía con su propio cluster frente a los demás. Otra alternativa que suele ser más robusta que los métodos anteriores es el Estadístico de Brecha (Gap Statistic), el cual compara la dispersión de los clusters con una distribución de referencia aleatoria.

### 1.4.2 Segmentación de Clientes con K-Means 🎁 [1 punto]

Por último, Mr. Lepin, impaciente de no entender lo que usted intenta explicarle, le solicita que por favor muestre algún resultado "visual y entendible" de los grupos encontrados.

En base a la elección de k realizada en la sección anterior, utilice este valor escogido y entrene un modelo de K-means utilizando el mismo pipeline de scikit-learn utilizado anteriormente.

Una vez ajustado los datos, genere una tabla con los promedios (o medianas) para cada uno de los atributos, agrupando estos por el clúster que pertenecen.

Finalmente, construya un heatmap de las características promedio de cada cluster para visualizar y comparar los perfiles de los grupos.

**Estadísticas de Referencia para K=6:**

Ud. debe calcularlas - Varían de ejecución en ejecución.


|         | Length  | Recency   | Frequency | Monetary | Periodicity |       |
|---------|---------|-----------|----------|-------------|-------|-------|
| Cluster |         |           |          |             |       |       |
|    0    |   258.8 |      45.2 |     76.1 |      1107.7 | 107.6 |   449 |
|    1    |    76.1 |     217.6 |     45.5 |       791.7 |  14.1 |   466 |
|    2    |   368.5 |       4.8 |   2715.0 |    226621.6 |   4.2 |     4 |
|    3    |    85.3 |      45.7 |     65.8 |      1047.0 |  10.5 |   987 |
|    4    |   347.2 |      15.9 |   1658.0 |     35829.3 |   8.0 |    25 |
|    5    |   298.0 |      29.8 |    183.8 |      3639.9 |  32.0 |  1188 |

In [17]:
# Calculamos K-Means
k_optimal = 3

pipeline_kmeans_final = Pipeline(
    [
        ("features", FunctionTransformer(custom_features)),
        ("scaler", scaler_transformer),
        ("kmeans", KMeans(n_clusters=k_optimal, random_state=42, n_init=10)),
    ]
)

In [18]:
# Entrenamos y obtenemos etiquetas
pipeline_kmeans_final.fit(df_retail)
kmeans_labels = pipeline_kmeans_final.predict(df_retail)

# Visualizamos los clusters en las tres proyecciones
plot_dim_reductions(pca_proj, tsne_proj, umap_proj, name=f"KMeans K={k_optimal}", colors=kmeans_labels).show()

In [19]:
# Generamos la tabla con promedios por cluster
df_with_clusters = df.copy()
df_with_clusters["Cluster"] = kmeans_labels

# Agrupamos por cluster y calculamos promedios
cluster_profiles = (
    df_with_clusters.groupby("Cluster")[["Length", "Recency", "Monetary", "Frequency", "Periodicity"]].mean().round(2)
)

# Agregamos la cantidad de clientes por cluster
cluster_profiles["Cantidad"] = df_with_clusters.groupby("Cluster").size()

# Mostramos la tabla
cluster_profiles

,Length,Recency,Monetary,Frequency,Periodicity,Cantidad
Cluster,,,,,,
0,41.00,54.93,372.96,47.88,5.47,1598
1,278.47,36.94,393.92,169.02,44.61,1759
2,23.46,251.80,349.03,28.35,3.47,957


In [20]:
# Obtenemos los datos escalados del pipeline
scaled_features = pipeline_kmeans_final.named_steps["scaler"].transform(
    pipeline_kmeans_final.named_steps["features"].transform(df_retail)
)
scaled_features_df = pd.DataFrame(scaled_features, columns=features)

# Calculamos el perfil medio de features escaladas por cluster de K-Means
cluster_profile = scaled_features_df.assign(cluster=kmeans_labels).groupby("cluster").mean().round(3)

# Renombramos los índices para una mejor visualización
cluster_profile.index = [f"{i}" for i in cluster_profile.index]

fig = px.imshow(
    cluster_profile,
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    text_auto=True,
    aspect="auto",
    title=f"Perfil medio de cada cluster (features escaladas, K-Means K={k_optimal})",
    labels={"x": "Feature", "y": "Cluster", "color": "Media escalada"},
    height=400,
    width=600,
)

fig.show()

### Preguntas sobre K-Means: 

- ¿Se separaron bien los distintos clusters en cada visualización? 

> Respuesta:
No, se observa en el gráfico que los clusters enocntrados no lograron separar bien los datos, ya que no existe una concordancia visual entre la forma de los clusters y la forma en que se proyectan los datos según las diferentes técnicas de reducción de dimensionalidad. Esto hace que algunos clusters parezcan creados casi de forma arbitraria, incluso llegando a existir sectores en donde hay un solapamiento de datos pertenecientes a clusters distintos.

- ¿Es posible observar agrupaciones coherentes?

> Respuesta

Sí, es posible identificar estructuras, aunque su claridad varía según la técnica. En PCA, los datos se concentran en una única formación triangular de alta densidad, lo que sugiere que las proyecciones lineales no bastan para separar los grupos. En contraste, UMAP y t-SNE revelan agrupaciones mucho más coherentes, caracterizadas por regiones de alta densidad separadas por espacios vacíos. Estas formaciones sugieren una estructura subyacente no lineal en los datos que sí permite hablar de clusters naturales, a diferencia de PCA.

- ¿Quedarían mejor más o menos clusters?

> Respuesta:

Ninguna de las opciones necesariamente ayudaría a interpretar los datos. Por una parte, aumentar el número de clusters reduciría la inercia, pero a costa de la interpretabilidad. Un exceso de grupos generaría fronteras artificiales en zonas de alta densidad, aumentando el solapamiento y dificultando la distinción de datos. Por otro lado, reducir el número a $k=2$ resultaría en una sobresimplificación, donde el algoritmo probablemente trazaría una división lineal arbitraria que no capturaría la estructura detectada en proyecciones como UMAP o t-SNE. Por lo tanto, el valor actual parece ser el equilibrio más razonable entre resolución de los datos y complejidad del modelo. 

- ¿K-Means, dada la forma de las proyecciones, será el mejor método para clusterizar este dataset?¿Habrá algún otro mejor?

> Respuesta:

Probablemente no sea el mejor método. K-Means asume que los clusters son esféricos y de tamaño similar, lo que "choca" con las estructuras irregulares y densas que revelan UMAP y t-SNE. Dado que estas proyecciones muestran agrupaciones densas y separadas por espacios vacíos, un algoritmo de clustering basado en densidad como DBSCAN o HDBSCAN sería mucho más adecuado. Estos métodos, a diferencia de K-Means, pueden identificar clusters de formas arbitrarias y filtrar el ruido (puntos aislados) sin forzarlos a pertenecer a un grupo.

Y por último:

- Nombre a cada uno de los clusters según el comportamiento de sus miembros (ej. "C1: Compran poco pero con gran valor...") - Si es necesario, ajuste el número de clusters antes de responder.

> Respuestas:

Según lo observado en el heatmap anteriormente presentado se pueden clasificar de la siguiente forma a los clientes analizados:

1. Clientes de Bajo Perfil o Recientes (C0): Presentan niveles mínimos en todas las variables. Son clientes que no han generado una relación duradera ni han realizado compras significativas aún.

2. Clientes de Larga Trayectoria (C1): Se definen por su alta puntuación en la variable Length. Son los miembros más antiguos del dataset, representando la base de clientes fidelizada a largo plazo.

3. Clientes Inactivos o con Riesgo de Abandono (C2): Su rasgo distintivo es el alto valor en Recency. Esto indica que ha transcurrido un periodo largo desde su última actividad, por lo que requieren estrategias de reactivación urgentes.

Justifique su respuesta y no decepcione a Mr. Lepin.

## 1.5 Detección de Anomalías con DBSCAN [1 punto]
En esta sección, utilizará el algoritmo DBSCAN para identificar posibles anomalías (outliers) en los clientes del retail.

- Puede aplicar DBSCAN sobre las características originales escaladas (**LRMFP**) o sobre alguna de las proyecciones 2D (PCA, t-SNE o UMAP). Justifique su elección en las preguntas al final de la sección.
- Visualice los resultados usando `plot_dim_reductions`, mostrando los clusters y resaltando los outliers (label = -1) en las tres proyecciones (PCA, t-SNE, UMAP).

In [21]:
from sklearn.neighbors import NearestNeighbors

# Calculamos distancias al k-ésimo vecino más cercano
k = 25
neighbors = NearestNeighbors(n_neighbors=k)
neighbors_fit = neighbors.fit(scaled_features_df)
distances, indices = neighbors_fit.kneighbors(scaled_features_df)

# Ordenamos distancias
distances = np.sort(distances[:, k - 1], axis=0)

# Graficamos k-distancias para encontrar un "codo" y obtener un posible valor de eps
fig = px.line(
    y=distances,
    title=f"K-distance Graph (k={k})",
    labels={"y": f"Distancia al {k}-ésimo vecino", "x": "Muestras (ordenadas)"},
    height=400,
)
fig.show()

In [22]:
from sklearn.cluster import DBSCAN  # noqa: F401

# Aplicamos DBSCAN a las características escaladas
dbscan = DBSCAN(eps=0.035, min_samples=25)
dbscan_labels = dbscan.fit_predict(scaled_features_df)

# Visualizamos en las tres proyecciones
fig_dbscan = plot_dim_reductions(
    pca_proj,
    tsne_proj,
    umap_proj,
    name="DBSCAN - Detección de Anomalías",
    colors=dbscan_labels,
)
fig_dbscan.show()

In [23]:
# Creamos dataframe con etiquetas DBSCAN
df_with_dbscan = df.copy()
df_with_dbscan["DBSCAN_Cluster"] = dbscan_labels

# Preparamos datos escalados con etiquetas
cluster_profile_dbscan = scaled_features_df.assign(cluster=dbscan_labels).groupby("cluster").mean().round(3)

# Renombramos índices para claridad
cluster_names = {-1: "Outliers"}
cluster_names.update({i: f"Cluster {i}" for i in np.unique(dbscan_labels) if i != -1})
cluster_profile_dbscan.index = cluster_profile_dbscan.index.map(cluster_names)

fig = px.imshow(
    cluster_profile_dbscan,
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0,
    text_auto=True,
    aspect="auto",
    title="Perfil de Clusters y Outliers (DBSCAN)",
    labels={"x": "Feature", "y": "Cluster/Outliers", "color": "Media escalada"},
    height=400,
    width=600,
)

fig.show()

### Preguntas sobre DBSCAN


1. ¿Por qué decidiste usar los datos originales completos o las proyecciones para aplicar DBSCAN? ¿Por qué no usaste la otra opción?

> Respuesta:

Se decidió aplicar DBSCAN sobre las proyecciones (t-SNE/UMAP) en lugar de los datos originales porque estos algoritmos de reducción de dimensionalidad agrupan los datos basándose en estructuras de vecindad local, revelando regiones de alta densidad y separaciones claras que no son evidentes en el espacio original de alta dimensión.

A su vez, no se utilizaron los datos originales completos principalmente por la "maldición de la dimensionalidad". En espacios de muchas dimensiones, la distancia entre puntos tiende a volverse uniforme, lo que dificulta que DBSCAN encuentre una densidad local significativa para definir los clusters.

2. ¿Cómo elegiste los parámetros de DBSCAN (`eps`, `min_samples`)? ¿Probaste diferentes valores? ¿Cómo afectó esto los resultados?

> Respuesta:

La elección de parámetros fue un proceso iterativo. Inicialmente, se utilizó el método del codo sobre el gráfico de $k$-distancias con un `min_samples = 10` para estimar `eps`. Sin embargo, los resultados fueron deficientes, ya que el algoritmo agrupaba la mayoría de los datos en un único cluster de gran tamaño, dejando puntos aislados como ruido (etiqueta $-1$). Tras experimentar sin éxito variando `eps` en ese rango, se decidió cambiar la estrategia. Se incrementó significativamente el `min_samples` a $25$ para forzar la formación de clusters más densos y robustos. Finalmente, mediante una búsqueda manual, se optó por un `eps = 0.035`, un valor considerablemente menor al sugerido por la k-distancia ($0.18$). Esta configuración permitió capturar de forma más precisa las estructuras locales observadas en las proyecciones, aunque con el compromiso de etiquetar una mayor cantidad de datos como ruido. Este ajuste priorizó la pureza de los clusters encontrados sobre la cobertura total del dataset.

3. ¿Tienen sentido los outliers encontrados según el contexto del negocio? ¿Qué interpretación le das a estos clientes? Analiza los datos con pandas si es necesario.

> Respuesta:

Creemos que los outliers encontrados sí pueden tener un sentido estratégico dentro del contexto del negocio (aunque con el matiz de que el proceso fue bastante arbitrario). Al analizar el mapa de calor, se observa que los outliers presentan un comportamiento híbrido: tienen una antigüedad considerable (`Length = 0.546`) y poseen valores relevantes para las variables `Periodicity` y `Recency`, pero no logran la densidad suficiente para formar un grupo homogéneo bajo los parámetros estrictos de DBSCAN. En este sentido, los outliers pueden representar clientes con comportamientos atípicos o en transición. A diferencia del Cluster 1 (clientes perdidos con altísimo `Recency`) o el Cluster 2 (clientes muy antiguos y leales), este perfil de clientes serían individuos que no encajan en los patrones estándar de compra.

# Conclusión
Eso ha sido todo para el lab de hoy, recuerden que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda del laboratorio, no duden en contactarnos por correo, Discord o U-cursos.

![Gracias Totales!](https://i.pinimg.com/originals/65/ae/27/65ae270df87c3c4adcea997e48f60852.gif "bruno")


<br>
<center>
<img src="https://i.kym-cdn.com/photos/images/original/001/194/195/b18.png" width=100 height=50 />
</center>
<br>